Import Libraries

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
import os

# Set display options for better visualization
pd.set_option('display.max_columns', None)

Load feature dataset

In [10]:
# Load the feature dataset
df = pd.read_csv('features_extracted.csv')

# Display initial dataset info
print("Initial Dataset Shape:", df.shape)
print("\nInitial Dataset Info:")
print(df.info())
print("\nFirst 5 rows of the dataset:")
print(df.head())

Initial Dataset Shape: (8, 9)

Initial Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   url                   8 non-null      object
 1   parameter             8 non-null      object
 2   input_value           4 non-null      object
 3   input_length          8 non-null      int64 
 4   num_special_chars     8 non-null      int64 
 5   num_digits            8 non-null      int64 
 6   param_type            8 non-null      object
 7   contains_sql_keyword  8 non-null      int64 
 8   contains_html_tag     8 non-null      int64 
dtypes: int64(5), object(4)
memory usage: 708.0+ bytes
None

First 5 rows of the dataset:
                               url   parameter  \
0  http://localhost:8080/login.php    username   
1  http://localhost:8080/login.php    password   
2  http://localhost:8080/login.php       Login   
3  http

Handle Missing Data

In [11]:
# Check for missing values
print("Missing Values Before Handling:")
print(df.isnull().sum())

Missing Values Before Handling:
url                     0
parameter               0
input_value             4
input_length            0
num_special_chars       0
num_digits              0
param_type              0
contains_sql_keyword    0
contains_html_tag       0
dtype: int64


In [12]:
# Handle missing values
# For numerical columns, fill with median
numerical_cols = ['input_length', 'num_special_chars', 'num_digits']
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

In [13]:
# For categorical columns, fill with mode
categorical_cols = ['url', 'parameter', 'param_type', 'input_value']
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [14]:
# For boolean-like columns, fill with False
boolean_cols = ['contains_sql_keyword', 'contains_html_tag']
for col in boolean_cols:
    df[col] = df[col].fillna(False)

In [15]:
# Verify no missing values remain
print("\nMissing Values After Handling:")
print(df.isnull().sum())


Missing Values After Handling:
url                     0
parameter               0
input_value             0
input_length            0
num_special_chars       0
num_digits              0
param_type              0
contains_sql_keyword    0
contains_html_tag       0
dtype: int64


Encode Categorical Columns

In [16]:
# Identify categorical columns
categorical_cols = ['url', 'parameter', 'param_type']

In [17]:
# Apply Label Encoding to categorical columns
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col])
    label_encoders[col] = le  # Save encoder for potential future use

In [18]:
# Drop original categorical columns
df = df.drop(categorical_cols, axis=1)

# For input_value, we'll assume it's not needed for ML (as it's raw input)
# If needed, we can encode it similarly or drop it
df = df.drop('input_value', axis=1)

In [19]:
# Verify the encoded dataset
print("\nDataset After Encoding:")
print(df.head())


Dataset After Encoding:
   input_length  num_special_chars  num_digits  contains_sql_keyword  \
0             3                  0           0                     0   
1             3                  0           0                     0   
2             5                  0           0                     0   
3            32                  0          20                     0   
4             3                  0           0                     0   

   contains_html_tag  url_encoded  parameter_encoded  param_type_encoded  
0                  0            0                  3                   3  
1                  0            0                  1                   1  
2                  0            0                  0                   2  
3                  0            0                  2                   0  
4                  0            0                  3                   3  


Normalize Numerical Columns

In [20]:
# Identify numerical columns
numerical_cols = ['input_length', 'num_special_chars', 'num_digits']

# Apply MinMaxScaler to normalize numerical columns to [0, 1]
scaler = MinMaxScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# Verify normalized values
print("\nDataset After Normalization:")
print(df.head())


Dataset After Normalization:
   input_length  num_special_chars  num_digits  contains_sql_keyword  \
0      0.000000                0.0    0.000000                     0   
1      0.000000                0.0    0.000000                     0   
2      0.068966                0.0    0.000000                     0   
3      1.000000                0.0    0.909091                     0   
4      0.000000                0.0    0.000000                     0   

   contains_html_tag  url_encoded  parameter_encoded  param_type_encoded  
0                  0            0                  3                   3  
1                  0            0                  1                   1  
2                  0            0                  0                   2  
3                  0            0                  2                   0  
4                  0            0                  3                   3  


Generate and Save the Cleaned Dataset

In [24]:
# Ensure all columns are numeric
print("\nDataset Info After Preprocessing:")
print(df.info())

# Verify no missing values
print("\nFinal Missing Values Check:")
print(df.isnull().sum())

# Save the cleaned dataset
df.to_csv('cleaned_dataset.csv', index=False)
print(f"\nCleaned dataset saved.")


Dataset Info After Preprocessing:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   input_length          8 non-null      float64
 1   num_special_chars     8 non-null      float64
 2   num_digits            8 non-null      float64
 3   contains_sql_keyword  8 non-null      int64  
 4   contains_html_tag     8 non-null      int64  
 5   url_encoded           8 non-null      int64  
 6   parameter_encoded     8 non-null      int64  
 7   param_type_encoded    8 non-null      int64  
dtypes: float64(3), int64(5)
memory usage: 644.0 bytes
None

Final Missing Values Check:
input_length            0
num_special_chars       0
num_digits              0
contains_sql_keyword    0
contains_html_tag       0
url_encoded             0
parameter_encoded       0
param_type_encoded      0
dtype: int64

Cleaned dataset saved.


In [25]:
# Display final dataset details
print("\nFinal Dataset Shape:", df.shape)
print("\nFinal Dataset Info:")
print(df.info())
print("\nFirst 5 Rows of Final Dataset:")
print(df.head())


Final Dataset Shape: (8, 8)

Final Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   input_length          8 non-null      float64
 1   num_special_chars     8 non-null      float64
 2   num_digits            8 non-null      float64
 3   contains_sql_keyword  8 non-null      int64  
 4   contains_html_tag     8 non-null      int64  
 5   url_encoded           8 non-null      int64  
 6   parameter_encoded     8 non-null      int64  
 7   param_type_encoded    8 non-null      int64  
dtypes: float64(3), int64(5)
memory usage: 644.0 bytes
None

First 5 Rows of Final Dataset:
   input_length  num_special_chars  num_digits  contains_sql_keyword  \
0      0.000000                0.0    0.000000                     0   
1      0.000000                0.0    0.000000                     0   
2      0.068966                0